In [1]:
import numpy as np
import pandas as pd
# neither year nor len are statistically significant predictors of sex
from pathlib import Path
import numpy as np
import pandas as pd

In [30]:
CWD = Path('.').absolute().resolve()
DATA_DIR = '.nlpia2-data'
DATA_FILE = 'baby-names-region.csv.gz'
CWD

PosixPath('/home/hobs/code/tangibleai/nlpia2/src/nlpia2/ch05')

In [31]:
parent = CWD
data_dir = parent / DATA_DIR 
filepath = data_dir / DATA_FILE
for i in range(10):
    print(filepath)
    if filepath.is_file():
        break
    parent = parent.parent
    data_dir = parent / DATA_DIR 
    filepath = data_dir / DATA_FILE
filepath

/home/hobs/code/tangibleai/nlpia2/src/nlpia2/ch05/.nlpia2-data/baby-names-region.csv.gz
/home/hobs/code/tangibleai/nlpia2/src/nlpia2/.nlpia2-data/baby-names-region.csv.gz
/home/hobs/code/tangibleai/nlpia2/src/.nlpia2-data/baby-names-region.csv.gz
/home/hobs/code/tangibleai/nlpia2/.nlpia2-data/baby-names-region.csv.gz


PosixPath('/home/hobs/code/tangibleai/nlpia2/.nlpia2-data/baby-names-region.csv.gz')

In [41]:
df = pd.read_csv(filepath)

In [42]:
np.random.seed(451)
df = df.sample(10_000)
df.head()

,region,sex,year,name,count,freq
6139665,WV,F,1987,Brittani,10,0.000003
2565339,MD,F,1954,Ida,18,0.000005
22297,AK,M,1988,Maxwell,5,0.000001
5114650,TN,F,1972,Charlene,24,0.000008
2126395,KS,M,1954,Todd,11,0.000003


In [43]:
names = df['name'].unique()
names[:10]

array(['Brittani', 'Ida', 'Maxwell', 'Charlene', 'Todd', 'Aubrey',
       'Arianna', 'Otis', 'Trenton', 'Faustino'], dtype=object)

In [44]:
istrain = np.random.rand(len(names)) < .9
istest = ~istrain
groups = df.groupby('name')
dfistrain = pd.DataFrame([istrain]).T
dfistrain.columns = ['istrain']
dfistrain.index = names
dfistrain.sum()


istrain    3642
dtype: int64

In [45]:
len(names)

4025

In [46]:
dfistrain = pd.DataFrame([istrain]).T
dfistrain

,0
0,True
1,True
2,True
3,True
4,True
...,...
4020,True
4021,True
4022,True
4023,True


In [47]:
dfistrain.columns = ['istrain']
dfistrain['name'] = names
# dfistrain.index = names
dfistrain

,istrain,name
0,True,Brittani
1,True,Ida
2,True,Maxwell
3,True,Charlene
4,True,Todd
...,...,...
4020,True,Aydan
4021,True,Iker
4022,True,Shonda
4023,True,Carley


In [48]:
istrain = df.merge(dfistrain, on='name')['istrain']
istrain

0       True
1       True
2       True
3       True
4       True
        ... 
9995    True
9996    True
9997    True
9998    True
9999    True
Name: istrain, Length: 10000, dtype: bool

In [49]:
istrain.sum() / len(istrain)

0.9086

In [53]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced')
df['len'] = df['name'].str.len()
X = df[['len']]
y = df['sex']
X.head()

,len
6139665,8
2565339,3
22297,7
5114650,8
2126395,4


In [54]:
model.fit(X,y)

LogisticRegression(class_weight='balanced')

In [55]:
model.score(X, y)

0.535

In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), lowercase=False)
vectorizer

TfidfVectorizer(analyzer='char', lowercase=False, ngram_range=(1, 3))

In [57]:
df.head()


,region,sex,year,name,count,freq,len
6139665,WV,F,1987,Brittani,10,0.000003,8
2565339,MD,F,1954,Ida,18,0.000005,3
22297,AK,M,1988,Maxwell,5,0.000001,7
5114650,TN,F,1972,Charlene,24,0.000008,8
2126395,KS,M,1954,Todd,11,0.000003,4


In [61]:
istrain.index = df.index
istrain.head()

6139665    True
2565339    True
22297      True
5114650    True
2126395    True
Name: istrain, dtype: bool

In [62]:
vectorizer.fit(df['name'][istrain])


TfidfVectorizer(analyzer='char', lowercase=False, ngram_range=(1, 3))

In [67]:
vectorizer.transform(['Amy']).todense()

matrix([[0.25502099, 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ]])

In [68]:
vecs = vectorizer.transform(df['name'])

In [69]:
dfvecs = pd.DataFrame.sparse.from_spmatrix(vecs)
dfvecs.head(2).T.head(100)

,0,1
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
...,...,...
95,0.0,0.0
96,0.0,0.0
97,0.0,0.0
98,0.0,0.0


In [70]:
dfvecs.columns = vectorizer.get_feature_names_out()
dfvecs.head(5).T.head(100)
df['name'].head()


6139665    Brittani
2565339         Ida
22297       Maxwell
5114650    Charlene
2126395        Todd
Name: name, dtype: object

In [71]:
dfvecs.index = df['name']
dfvecs.head(5).T.head(100)


name,Brittani,Ida,Maxwell,Charlene,Todd
A,0.0,0.0,0.0,0.0,0.0
Aa,0.0,0.0,0.0,0.0,0.0
Aad,0.0,0.0,0.0,0.0,0.0
Aah,0.0,0.0,0.0,0.0,0.0
Aal,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
Asi,0.0,0.0,0.0,0.0,0.0
Asp,0.0,0.0,0.0,0.0,0.0
At,0.0,0.0,0.0,0.0,0.0
Ata,0.0,0.0,0.0,0.0,0.0


In [72]:
dfvecs['Amy'].head(5).T.head(100)


name
Brittani    0.0
Ida         0.0
Maxwell     0.0
Charlene    0.0
Todd        0.0
Name: Amy, dtype: Sparse[float64, 0]

In [73]:
X = dfvecs
X.head()

,A,Aa,Aad,Aah,Aal,Aan,Aar,Ab,Abb,Abd,...,zmi,zo,zr,zra,zv,zvi,zy,zz,zze,zzi
name,,,,,,,,,,,,,,,,,,,,,
Brittani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Ida,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Maxwell,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Charlene,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Todd,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [75]:
X.index = df['name']
model.fit(X[istrain], y[istrain])

/tmp/ipykernel_6921/2036506220.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  model.fit(X[istrain], y[istrain])


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).